# Tahap 5: Model Evaluation
**Mata Kuliah:** Penalaran Komputer — SubCPMK-3 Case-Based Reasoning  
**Domain:** Perlindungan Anak  
**Tujuan:** Ukur dan analisis performa retrieval & prediksi secara lengkap

### Alur Notebook:
1. Install & Import
2. Load Semua Artefak
3. Evaluasi Retrieval (Accuracy, Precision, Recall, F1)
4. Evaluasi Prediksi Solution Reuse
5. Perbandingan TF-IDF vs Embedding
6. Analisis Kegagalan (Error Analysis)
7. Visualisasi Performa
8. Simpan Laporan Metrik
9. Ringkasan & Rekomendasi

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q scikit-learn sentence-transformers pandas numpy joblib matplotlib seaborn

## Cell 2 — Import Library

In [ ]:
import json
import re
import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
from collections import Counter

import joblib
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import cross_val_score
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["font.size"]  = 10

print("✅ Library berhasil diimport")

## Cell 3 — Konfigurasi

In [ ]:
CONFIG = {
    "TOP_K"          : 5,
    "RANDOM_STATE"   : 42,
}

BASE_DIR    = Path(".").resolve()
DATA_PROC   = BASE_DIR / "data" / "processed"
DATA_EVAL   = BASE_DIR / "data" / "eval"
DATA_RESULT = BASE_DIR / "data" / "results"
MODEL_DIR   = BASE_DIR / "models"
LOGS_DIR    = BASE_DIR / "logs"

DATA_EVAL.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s | %(levelname)s | %(message)s",
    handlers = [
        logging.FileHandler(LOGS_DIR / "evaluation.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
print("✅ Konfigurasi selesai")

## Cell 4 — Load Semua Artefak

In [ ]:
# ── Cases ──────────────────────────────────────────────────
with open(DATA_PROC / "cases.json", encoding="utf-8") as f:
    cases = json.load(f)
df = pd.DataFrame(cases)
print(f"✅ {len(df)} kasus dimuat")

# ── TF-IDF ─────────────────────────────────────────────────
tfidf   = joblib.load(MODEL_DIR / "tfidf_vectorizer.pkl")
X_tfidf = joblib.load(MODEL_DIR / "tfidf_matrix.pkl")
svm     = joblib.load(MODEL_DIR / "svm_model.pkl")
le      = joblib.load(MODEL_DIR / "label_encoder.pkl")
print(f"✅ TF-IDF + SVM dimuat — matrix: {X_tfidf.shape}")

# ── Embeddings ─────────────────────────────────────────────
embeddings = np.load(MODEL_DIR / "embeddings.npy")
print(f"✅ Embeddings dimuat — shape: {embeddings.shape}")

ST_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"\n🔄 Loading SentenceTransformer...")
st_model = SentenceTransformer(ST_MODEL)
print(f"✅ ST model siap")

# ── Predictions dari Tahap 4 ───────────────────────────────
pred_path = DATA_RESULT / "predictions.csv"
if pred_path.exists():
    df_pred = pd.read_csv(pred_path)
    print(f"✅ predictions.csv dimuat — {len(df_pred)} baris")
else:
    df_pred = pd.DataFrame()
    print("⚠️  predictions.csv tidak ditemukan — evaluasi prediksi akan dilewati")

# ── Queries eval ───────────────────────────────────────────
queries_path = DATA_EVAL / "queries.json"
if queries_path.exists():
    with open(queries_path, encoding="utf-8") as f:
        queries_eval = json.load(f)
    print(f"✅ queries.json dimuat — {len(queries_eval)} query")
else:
    queries_eval = []
    print("⚠️  queries.json tidak ditemukan")

## Cell 5 — Fungsi Pendukung (Retrieve & Preprocessing)

In [ ]:
STOPWORDS_ID = set([
    "yang","dan","di","ke","dari","dengan","untuk","pada","dalam",
    "adalah","ini","itu","atau","juga","sudah","oleh","tidak","ada",
    "bahwa","telah","akan","maka","serta","para","kepada","karena",
    "atas","sebagai","dapat","tersebut","sehingga","antara","nomor",
    "halaman","republik","indonesia","mahkamah","pengadilan","agung",
])

def preprocess_teks(teks: str, max_chars: int = 8000) -> str:
    if not teks or not isinstance(teks, str): return ""
    teks = teks[:max_chars].lower()
    teks = re.sub(r"\d+", " ", teks)
    teks = re.sub(r"[^a-z\s]", " ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    kata = [k for k in teks.split() if k not in STOPWORDS_ID and len(k) > 2]
    return " ".join(kata)

def buat_teks_representasi(row) -> str:
    bagian = []
    for field in ["pasal_dakwaan","pasal_terbukti","amar_putusan"]:
        val = str(row.get(field,"")).strip()
        if val: bagian.append(val); bagian.append(val)
    for field in ["ringkasan_fakta","barang_bukti","terdakwa"]:
        val = str(row.get(field,"")).strip()
        if val: bagian.append(val)
    teks_penuh = str(row.get("teks_full",""))[:3000]
    if teks_penuh: bagian.append(teks_penuh)
    return " ".join(bagian)

def retrieve(query: str, k: int = 5, mode: str = "tfidf") -> list:
    query_clean = preprocess_teks(query)
    if not query_clean: return []
    if mode == "tfidf":
        q_vec  = tfidf.transform([query_clean])
        scores = cosine_similarity(q_vec, X_tfidf).flatten()
    else:
        q_emb  = st_model.encode([query], normalize_embeddings=True)
        scores = cosine_similarity(q_emb, embeddings).flatten()
    top_idx = np.argsort(scores)[::-1][:k]
    return [{"case_id": df.iloc[i]["case_id"],
             "score"  : round(float(scores[i]), 4),
             "label"  : str(df.iloc[i].get("label",""))}
            for i in top_idx]

# Buat label kolom jika belum ada
def buat_label(amar: str) -> str:
    amar = str(amar).lower()
    if any(k in amar for k in ["bebas","tidak terbukti","dibebaskan"]): return "bebas"
    elif any(k in amar for k in ["lepas","tuntutan"]): return "lepas"
    elif any(k in amar for k in ["terbukti","bersalah","pidana","penjara","denda"]): return "terbukti"
    return "lainnya"

if "label" not in df.columns:
    df["label"] = df["amar_putusan"].apply(buat_label)

if "teks_repr" not in df.columns:
    df["teks_repr"]      = df.apply(buat_teks_representasi, axis=1)
    df["teks_processed"] = df["teks_repr"].apply(preprocess_teks)

print("✅ Fungsi pendukung siap")

## Cell 6 — Evaluasi Model SVM (Classification)

> Evaluasi SVM sebagai classifier menggunakan cross-validation  
> karena data mungkin terlalu kecil untuk split yang stabil.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

print(f"{'='*55}")
print(f"  EVALUASI SVM — CROSS VALIDATION")
print(f"{'='*55}\n")

# Siapkan data
label_counts = df["label"].value_counts()
label_valid  = label_counts[label_counts >= 2].index.tolist()
df_svm       = df[df["label"].isin(label_valid)].copy()

from sklearn.preprocessing import LabelEncoder
le_eval = LabelEncoder()
y_all   = le_eval.fit_transform(df_svm["label"])
idx_all = df_svm.index.tolist()
X_all   = X_tfidf[idx_all]

print(f"  Jumlah sampel : {len(df_svm)}")
print(f"  Label         : {label_valid}")
print(f"  Distribusi    :")
for l, c in label_counts.items():
    print(f"    {l:<15}: {c}")

# Cross-validation (5-fold jika cukup, else 3-fold)
n_splits = min(5, min(label_counts[label_valid]))
n_splits = max(2, n_splits)

print(f"\n  Cross-validation: {n_splits}-fold")

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=CONFIG["RANDOM_STATE"])
cv_results = cross_validate(
    svm, X_all, y_all, cv=cv,
    scoring=["accuracy","precision_macro","recall_macro","f1_macro"],
    return_train_score=False,
)

metrics_svm = {
    "accuracy"        : round(cv_results["test_accuracy"].mean(), 4),
    "precision_macro" : round(cv_results["test_precision_macro"].mean(), 4),
    "recall_macro"    : round(cv_results["test_recall_macro"].mean(), 4),
    "f1_macro"        : round(cv_results["test_f1_macro"].mean(), 4),
    "n_splits"        : n_splits,
    "n_samples"       : len(df_svm),
}

print(f"\n  Hasil Cross-Validation (mean):")
print(f"  {'Accuracy':<20}: {metrics_svm['accuracy']:.4f}")
print(f"  {'Precision (macro)':<20}: {metrics_svm['precision_macro']:.4f}")
print(f"  {'Recall (macro)':<20}: {metrics_svm['recall_macro']:.4f}")
print(f"  {'F1 (macro)':<20}: {metrics_svm['f1_macro']:.4f}")

# Train pada semua data untuk confusion matrix
svm.fit(X_all, y_all)
y_pred_all = svm.predict(X_all)
print(f"\n  Classification Report (train set):")
print(classification_report(y_all, y_pred_all,
                             target_names=le_eval.classes_,
                             zero_division=0))
logger.info(f"SVM CV: acc={metrics_svm['accuracy']}, f1={metrics_svm['f1_macro']}")

## Cell 7 — Evaluasi Retrieval: Precision@K, Recall@K, F1@K

> Evaluasi retrieval menggunakan **leave-one-out**:  
> setiap dokumen dijadikan query, lalu cek apakah dokumen dengan  
> label sama muncul di top-K hasil.

In [ ]:
def eval_retrieval_leave_one_out(mode: str = "tfidf", k: int = 5) -> dict:
    """
    Leave-one-out evaluation untuk retrieval.
    Untuk setiap dokumen i:
      - Jadikan dokumen i sebagai query
      - Ambil top-k dari sisa dokumen
      - Hitung berapa yang labelnya sama (relevan)
    """
    precision_list = []
    recall_list    = []
    f1_list        = []

    for i, row in df.iterrows():
        query_label = row.get("label", "")
        if query_label in ("", "lainnya"): continue

        # Jumlah dokumen relevan di seluruh corpus (selain diri sendiri)
        n_relevan_total = sum(
            1 for _, r in df.iterrows()
            if r["case_id"] != row["case_id"] and r.get("label","") == query_label
        )
        if n_relevan_total == 0: continue

        # Retrieve
        hasil = retrieve(
            str(row.get("teks_repr", row.get("teks_full",""))),
            k=k, mode=mode
        )
        # Hapus diri sendiri jika muncul
        hasil = [h for h in hasil if h["case_id"] != row["case_id"]][:k]
        if not hasil: continue

        # Hitung relevan di top-k
        n_relevan_topk = sum(1 for h in hasil if h["label"] == query_label)

        precision = n_relevan_topk / len(hasil)
        recall    = n_relevan_topk / min(n_relevan_total, k)
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0)

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)

    if not precision_list:
        return {"precision": 0, "recall": 0, "f1": 0, "n_queries": 0}

    return {
        "mode"      : mode,
        "k"         : k,
        "n_queries" : len(precision_list),
        "precision" : round(np.mean(precision_list), 4),
        "recall"    : round(np.mean(recall_list), 4),
        "f1"        : round(np.mean(f1_list), 4),
        "precision_std": round(np.std(precision_list), 4),
        "recall_std"   : round(np.std(recall_list), 4),
        "f1_std"       : round(np.std(f1_list), 4),
    }


print("🔄 Evaluasi retrieval TF-IDF (leave-one-out)...")
metrics_tfidf = eval_retrieval_leave_one_out(mode="tfidf", k=CONFIG["TOP_K"])

print("🔄 Evaluasi retrieval Embedding...")
metrics_embed = eval_retrieval_leave_one_out(mode="embedding", k=CONFIG["TOP_K"])

print(f"\n{'='*60}")
print(f"  HASIL EVALUASI RETRIEVAL — Leave-One-Out @ K={CONFIG['TOP_K']}")
print(f"{'='*60}")
print(f"  {'Metrik':<22} {'TF-IDF':>10} {'Embedding':>12}")
print(f"  {'─'*48}")
for m in ["precision","recall","f1"]:
    print(f"  {m.capitalize():<22} {metrics_tfidf[m]:>10.4f} {metrics_embed[m]:>12.4f}")
print(f"  {'─'*48}")
print(f"  {'N queries':<22} {metrics_tfidf['n_queries']:>10} {metrics_embed['n_queries']:>12}")
print(f"{'='*60}")

logger.info(f"Retrieval TF-IDF: P={metrics_tfidf['precision']}, R={metrics_tfidf['recall']}, F1={metrics_tfidf['f1']}")
logger.info(f"Retrieval Embed : P={metrics_embed['precision']}, R={metrics_embed['recall']}, F1={metrics_embed['f1']}")

## Cell 8 — Evaluasi Prediksi Solution Reuse

In [ ]:
if df_pred.empty:
    print("⚠️  predictions.csv kosong — jalankan Tahap 4 terlebih dahulu")
else:
    print(f"{'='*55}")
    print(f"  EVALUASI PREDIKSI SOLUTION REUSE")
    print(f"{'='*55}")

    # Filter baris yang punya label_sebenarnya
    df_eval_pred = df_pred[
        df_pred["label_sebenarnya"].notna() &
        df_pred["predicted_label"].notna() &
        (df_pred["label_sebenarnya"] != "") &
        (df_pred["predicted_label"] != "")
    ].copy()

    if len(df_eval_pred) == 0:
        print("⚠️  Tidak ada data evaluasi dengan label lengkap")
    else:
        y_true = df_eval_pred["label_sebenarnya"].tolist()
        y_pred_mv = df_eval_pred["majority_vote_label"].tolist()
        y_pred_ws = df_eval_pred["weighted_sim_label"].tolist()

        labels_unik = sorted(set(y_true + y_pred_mv + y_pred_ws))

        def hitung_metrik(y_true, y_pred, nama):
            avg = "macro" if len(set(y_true)) > 2 else "binary"
            pos = labels_unik[0] if avg == "binary" else None
            acc = accuracy_score(y_true, y_pred)
            try:
                p = precision_score(y_true, y_pred, average=avg,
                                    pos_label=pos, zero_division=0)
                r = recall_score(y_true, y_pred, average=avg,
                                 pos_label=pos, zero_division=0)
                f = f1_score(y_true, y_pred, average=avg,
                             pos_label=pos, zero_division=0)
            except Exception:
                p = r = f = 0.0
            print(f"\n  [{nama}]")
            print(f"    Accuracy : {acc:.4f}")
            print(f"    Precision: {p:.4f}")
            print(f"    Recall   : {r:.4f}")
            print(f"    F1-score : {f:.4f}")
            return {"nama": nama, "accuracy": round(acc,4),
                    "precision": round(p,4), "recall": round(r,4), "f1": round(f,4)}

        metrics_mv = hitung_metrik(y_true, y_pred_mv, "Majority Vote")
        metrics_ws = hitung_metrik(y_true, y_pred_ws, "Weighted Similarity")

        print(f"\n  Classification Report — Weighted Similarity:")
        print(classification_report(y_true, y_pred_ws,
                                     labels=labels_unik,
                                     zero_division=0))

        metrics_prediksi = [metrics_mv, metrics_ws]
        logger.info(f"Prediksi MV : acc={metrics_mv['accuracy']}, f1={metrics_mv['f1']}")
        logger.info(f"Prediksi WS : acc={metrics_ws['accuracy']}, f1={metrics_ws['f1']}")

## Cell 9 — Confusion Matrix SVM

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Confusion Matrix SVM ────────────────────────────────────
cm_svm = confusion_matrix(y_all, y_pred_all)
disp   = ConfusionMatrixDisplay(confusion_matrix=cm_svm,
                                 display_labels=le_eval.classes_)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix — SVM (Train Set)")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

# ── Distribusi label corpus ─────────────────────────────────
label_dist = df["label"].value_counts()
axes[1].bar(label_dist.index, label_dist.values,
            color=sns.color_palette("Blues_d", len(label_dist)))
axes[1].set_title("Distribusi Label di Corpus")
axes[1].set_xlabel("Label")
axes[1].set_ylabel("Jumlah Kasus")
for i, (l, v) in enumerate(label_dist.items()):
    axes[1].text(i, v + 0.2, str(v), ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(DATA_EVAL / "plot_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"💾 Plot disimpan: {DATA_EVAL / 'plot_confusion_matrix.png'}")

## Cell 10 — Perbandingan TF-IDF vs Embedding

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

metrik_names = ["Precision", "Recall", "F1"]
vals_tfidf   = [metrics_tfidf["precision"], metrics_tfidf["recall"], metrics_tfidf["f1"]]
vals_embed   = [metrics_embed["precision"], metrics_embed["recall"], metrics_embed["f1"]]

x = np.arange(len(metrik_names))
w = 0.35

# ── Bar chart perbandingan ──────────────────────────────────
bars1 = axes[0].bar(x - w/2, vals_tfidf, w, label="TF-IDF",    color="#4C72B0")
bars2 = axes[0].bar(x + w/2, vals_embed, w, label="Embedding", color="#55A868")

axes[0].set_title(f"Retrieval Performance @ K={CONFIG['TOP_K']}")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrik_names)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel("Score")
axes[0].legend()
axes[0].axhline(y=0.5, color="gray", linestyle="--", alpha=0.4)

for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{bar.get_height():.3f}", ha="center", fontsize=9)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{bar.get_height():.3f}", ha="center", fontsize=9)

# ── Radar chart SVM metrics ─────────────────────────────────
svm_metrik = [metrics_svm["accuracy"], metrics_svm["precision_macro"],
              metrics_svm["recall_macro"], metrics_svm["f1_macro"]]
svm_labels = ["Accuracy", "Precision", "Recall", "F1"]

theta  = np.linspace(0, 2*np.pi, len(svm_labels), endpoint=False).tolist()
vals_r = svm_metrik + [svm_metrik[0]]
theta += [theta[0]]

ax2 = plt.subplot(122, polar=True)
ax2.plot(theta, vals_r, "o-", linewidth=2, color="#4C72B0")
ax2.fill(theta, vals_r, alpha=0.25, color="#4C72B0")
ax2.set_xticks(theta[:-1])
ax2.set_xticklabels(svm_labels, fontsize=10)
ax2.set_ylim(0, 1)
ax2.set_title("SVM Performance (CV)", pad=15)
for angle, val in zip(theta[:-1], vals_r[:-1]):
    ax2.text(angle, val + 0.08, f"{val:.3f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(DATA_EVAL / "plot_perbandingan_model.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"💾 Plot disimpan: {DATA_EVAL / 'plot_perbandingan_model.png'}")

## Cell 11 — Analisis Kegagalan (Error Analysis)

> Identifikasi pola kasus yang sering salah diklasifikasi  
> dan berikan rekomendasi perbaikan.

In [ ]:
print(f"{'='*60}")
print(f"  ANALISIS KEGAGALAN — ERROR ANALYSIS")
print(f"{'='*60}\n")

# ── Kasus salah klasifikasi oleh SVM ───────────────────────
salah_svm = []
for i, (true, pred) in enumerate(zip(y_all, y_pred_all)):
    if true != pred:
        idx    = idx_all[i]
        row    = df.loc[idx]
        salah_svm.append({
            "case_id"     : row["case_id"],
            "label_benar" : le_eval.inverse_transform([true])[0],
            "label_pred"  : le_eval.inverse_transform([pred])[0],
            "amar_putusan": str(row.get("amar_putusan",""))[:100],
            "jumlah_kata" : row.get("jumlah_kata", 0),
            "n_kata_kunci": row.get("n_kata_kunci_pa", 0),
        })

df_salah = pd.DataFrame(salah_svm)

print(f"  Total kasus salah (SVM) : {len(df_salah)} / {len(y_all)}")
print(f"  Error rate              : {len(df_salah)/len(y_all):.2%}\n")

if not df_salah.empty:
    print(f"  Pola kesalahan (true → pred):")
    for _, r in df_salah.iterrows():
        print(f"    {r['case_id']}: {r['label_benar']} → {r['label_pred']}")
        print(f"      amar: {r['amar_putusan'][:80]}")

    print(f"\n  Statistik kasus salah:")
    print(f"    Avg jumlah kata  : {df_salah['jumlah_kata'].mean():.0f}")
    print(f"    Avg kata kunci PA: {df_salah['n_kata_kunci'].mean():.1f}")

    # Analisis akar masalah
    print(f"\n  Kemungkinan penyebab kesalahan:")
    for _, r in df_salah.iterrows():
        amar = r["amar_putusan"].lower()
        if len(amar) < 30:
            print(f"    [{r['case_id']}] Amar putusan terlalu pendek/kosong → metadata tidak lengkap")
        elif r["n_kata_kunci"] == 0:
            print(f"    [{r['case_id']}] Tidak ada kata kunci PA → kemungkinan dokumen tidak relevan domain")
        elif r["jumlah_kata"] < 500:
            print(f"    [{r['case_id']}] Dokumen pendek ({r['jumlah_kata']} kata) → teks tidak cukup informatif")
        else:
            print(f"    [{r['case_id']}] Ambiguitas label — amar putusan memiliki ciri dua label")
else:
    print("  ✅ Tidak ada kesalahan klasifikasi pada train set")

# ── Analisis retrieval rejection ───────────────────────────
print(f"\n{'─'*55}")
print(f"  ANALISIS REJECTION RETRIEVAL")
print(f"{'─'*55}")

skor_rendah = []
for i, row in df.iterrows():
    teks = str(row.get("teks_repr", row.get("teks_full","")))
    hasil = retrieve(teks, k=1, mode="tfidf")
    hasil = [h for h in hasil if h["case_id"] != row["case_id"]]
    if hasil and hasil[0]["score"] < 0.05:
        skor_rendah.append({
            "case_id"   : row["case_id"],
            "max_score" : hasil[0]["score"],
            "label"     : row.get("label",""),
        })

print(f"  Kasus dengan max similarity < 0.05: {len(skor_rendah)}")
if skor_rendah:
    print(f"  Daftar:")
    for s in skor_rendah[:5]:
        print(f"    {s['case_id']}: score={s['max_score']:.4f}, label={s['label']}")
    print(f"  Rekomendasi: kasus ini mungkin outlier atau teks terlalu pendek.")

## Cell 12 — Rekomendasi Perbaikan

In [ ]:
print(f"{'='*60}")
print(f"  REKOMENDASI PERBAIKAN MODEL")
print(f"{'='*60}\n")

# Bandingkan TF-IDF vs Embedding
f1_tfidf = metrics_tfidf["f1"]
f1_embed = metrics_embed["f1"]
mode_terbaik = "TF-IDF" if f1_tfidf >= f1_embed else "Embedding"

rekomendasi = []

# 1. Mode retrieval
rekomendasi.append({
    "aspek"    : "Mode Retrieval",
    "temuan"   : f"TF-IDF F1={f1_tfidf:.3f}, Embedding F1={f1_embed:.3f}",
    "saran"    : f"Gunakan {mode_terbaik} sebagai mode utama. "
                 f"{'Embedding lebih baik untuk teks panjang dan variasi kata.' if mode_terbaik == 'Embedding' else 'TF-IDF sudah cukup efisien untuk corpus ini.'}"
})

# 2. Ukuran corpus
if len(df) < 50:
    rekomendasi.append({
        "aspek"  : "Ukuran Corpus",
        "temuan" : f"Hanya {len(df)} dokumen",
        "saran"  : "Tambah corpus minimal ke 50+ dokumen untuk hasil yang lebih stabil. "
                   "Kasus dengan label 'lainnya' perlu dikategorikan ulang secara manual."
    })

# 3. Kualitas metadata
pct_amar_kosong = (df["amar_putusan"].astype(str).str.strip() == "").mean()
if pct_amar_kosong > 0.3:
    rekomendasi.append({
        "aspek"  : "Metadata Amar Putusan",
        "temuan" : f"{pct_amar_kosong:.0%} dokumen tidak ada amar putusan",
        "saran"  : "Perbaiki ekstraksi di Tahap 2 — coba prompt Groq yang lebih spesifik "
                   "atau tambahkan pola regex untuk bagian MENGADILI."
    })

# 4. Keseimbangan label
label_dist_val = df["label"].value_counts()
if label_dist_val.max() / label_dist_val.sum() > 0.7:
    rekomendasi.append({
        "aspek"  : "Imbalanced Labels",
        "temuan" : f"Label dominan: {label_dist_val.idxmax()} ({label_dist_val.max()/label_dist_val.sum():.0%})",
        "saran"  : "Dataset tidak seimbang. Pertimbangkan oversampling (SMOTE) "
                   "atau gunakan class_weight='balanced' di SVM."
    })

# 5. Error rate
error_rate = len(df_salah) / len(y_all) if len(y_all) > 0 else 0
if error_rate > 0.2:
    rekomendasi.append({
        "aspek"  : "SVM Error Rate",
        "temuan" : f"Error rate {error_rate:.0%}",
        "saran"  : "Coba tuning hyperparameter C di SVM, atau ganti ke RBF kernel. "
                   "Alternatif: gunakan Naive Bayes untuk dataset kecil."
    })

print(f"  {'No':<4} {'Aspek':<25} {'Temuan':<30}")
print(f"  {'─'*65}")
for i, r in enumerate(rekomendasi, 1):
    print(f"  {i:<4} {r['aspek']:<25} {r['temuan'][:30]}")
    print(f"       → {r['saran']}")
    print()

if not rekomendasi:
    print("  ✅ Tidak ada isu kritis yang ditemukan — model sudah cukup baik!")

## Cell 13 — Simpan Laporan Metrik Lengkap

In [ ]:
laporan = {
    "svm_classification": metrics_svm,
    "retrieval_tfidf"   : metrics_tfidf,
    "retrieval_embedding": metrics_embed,
    "mode_terbaik"      : mode_terbaik,
    "n_kasus_corpus"    : len(df),
    "distribusi_label"  : df["label"].value_counts().to_dict(),
    "error_analysis"    : {
        "n_salah_svm"       : len(df_salah),
        "error_rate_svm"    : round(len(df_salah)/len(y_all), 4) if len(y_all) > 0 else 0,
        "n_rejection"       : len(skor_rendah),
    },
    "rekomendasi"       : rekomendasi,
}

# Tambahkan metrik prediksi jika tersedia
if not df_pred.empty and "metrics_prediksi" in dir():
    laporan["prediksi_solution_reuse"] = metrics_prediksi

# Simpan
retrieval_metrics_path = DATA_EVAL / "retrieval_metrics.csv"
pd.DataFrame([metrics_tfidf, metrics_embed]).to_csv(
    retrieval_metrics_path, index=False)
print(f"✅ retrieval_metrics.csv → {retrieval_metrics_path}")

pred_metrics_path = DATA_EVAL / "prediction_metrics.csv"
if not df_pred.empty and "metrics_prediksi" in dir():
    pd.DataFrame(metrics_prediksi).to_csv(pred_metrics_path, index=False)
    print(f"✅ prediction_metrics.csv → {pred_metrics_path}")

full_report_path = DATA_EVAL / "full_report.json"
with open(full_report_path, "w", encoding="utf-8") as f:
    json.dump(laporan, f, ensure_ascii=False, indent=2)
print(f"✅ full_report.json → {full_report_path}")

print(f"\n{'='*55}")
print(f"  RINGKASAN AKHIR")
print(f"{'='*55}")
print(f"  Corpus          : {len(df)} kasus")
print(f"  SVM Accuracy    : {metrics_svm['accuracy']:.4f}")
print(f"  SVM F1 (macro)  : {metrics_svm['f1_macro']:.4f}")
print(f"  Retrieval TF-IDF F1@{CONFIG['TOP_K']} : {metrics_tfidf['f1']:.4f}")
print(f"  Retrieval Embed F1@{CONFIG['TOP_K']}  : {metrics_embed['f1']:.4f}")
print(f"  Mode terbaik    : {mode_terbaik}")
print(f"{'='*55}")
print(f"\n✅ Semua file evaluasi tersimpan di: {DATA_EVAL}")

---
## ✅ Tahap 5 Selesai — Pipeline CBR Lengkap!

**Output yang dihasilkan:**
```
/data/eval/
├── retrieval_metrics.csv        ← Precision, Recall, F1 retrieval
├── prediction_metrics.csv       ← Metrik prediksi solution reuse
├── full_report.json             ← Laporan lengkap semua metrik
├── plot_confusion_matrix.png    ← Confusion matrix SVM
└── plot_perbandingan_model.png  ← Bar chart TF-IDF vs Embedding
```

**Siklus CBR yang sudah selesai:**
```
Build Case Base  ✅  →  Case Representation  ✅
       ↑                        ↓
   Retain  ✅           Case Retrieval  ✅
       ↑                        ↓
   Revise  ✅  ←   Solution Reuse  ✅
```

**Checklist pengumpulan tugas:**
- [ ] GitHub repository publik dengan struktur `/data/`, `/notebooks/`, `README.md`
- [ ] Link repository diupload ke LMS
- [ ] README berisi cara install dan cara jalankan pipeline end-to-end